In [ ]:
"""
Tic-Tac-Toe (Human vs Agent) using Alpha-Beta Pruning
=====================================================

Genral WorkFlow Structure:
- method to place X/O on the board
- method to check winning condition
- method to alternate moves between human and agent
- alpha-beta pruning method (agent chooses next move)
- utility function (used by alpha-beta)

"""

# -------------------------
# Global constants
# -------------------------
EMPTY = " "
HUMAN = "X"
AGENT = "O"

# Agent look-ahead depth in plies
LOOKAHEAD = 4

WIN_LINES = [
    # Rows
    (0, 1, 2),
    (3, 4, 5),
    (6, 7, 8),
    # Columns
    (0, 3, 6),
    (1, 4, 7),
    (2, 5, 8),
    # Diagonals
    (0, 4, 8),
    (2, 4, 6),
]

In [ ]:
# =========================================================
# 1) Method: place X or O in the board
# =========================================================
def place_mark(board, index, mark):
    """
    Places 'mark' (X or O) at board[index] if the move is legal.
    Returns True if placed successfully, False otherwise.
    """
    if index < 0 or index > 8:
        return False

    if board[index] != EMPTY:
        return False

    board[index] = mark
    return True

In [ ]:
# =========================================================
# 2) Method: check winning condition
# =========================================================
def check_winner(board):
    """
    Returns:
      "X" if X won
      "O" if O won
      None if no winner
    """
    for line in WIN_LINES:
        a = line[0]
        b = line[1]
        c = line[2]

        if board[a] != EMPTY:
            if board[a] == board[b] and board[b] == board[c]:
                return board[a]

    return None


def is_draw(board):
    """
    Returns True if board is full and nobody has won.
    """
    if check_winner(board) is not None:
        return False

    for i in range(0, 9):
        if board[i] == EMPTY:
            return False

    return True


def get_available_moves(board):
    """
    Returns a list of indices where moves can be made.
    """
    moves = []
    for i in range(0, 9):
        if board[i] == EMPTY:
            moves.append(i)
    return moves



In [ ]:
# =========================================================
# Utility function for alpha-beta
# =========================================================
def utility(board):
    """
    Utility (score) from the perspective of the AGENT (O).

    Terminal scoring:
      - Agent (O) wins => +100
      - Human (X) wins => -100
      - Draw => 0

    Non-terminal heuristic:
      We count "potential lines" for each side.
      Scoring:
        2 in a row + 1 empty => +/- 10   (positive = good for Agent O, negative = good for Human X)
        1 in a row + 2 empty => +/- 1
    """
    score=0
    for line in WIN_LINES:
      for idx in line:

        AC=0
        HC=0
        EC=0

        if board(idx)==AGENT:
          AC+=1
        elif board(idx)==HUMAN:
          HC+=1
        else:
          EC+=1

        #Non-terminal utility score
        if AC==HC and EC==3:  #***
          score+=0
        elif AC==HC and HC==EC:  #XO*
          score+=0
        elif AC==2 and EC==1:  #OO*
          score+=10
        elif AC==2 and EC==1:  #O**
          score+=1
        elif HC==2 and EC==1:  #XX*
          score-=10
        elif HC==2 and EC==1:  #X**
          score-=1
        #Terminal utility score
        elif AC==3:
          score+=100
        elif HC==3:
          score-=100

      return score

In [ ]:
# =========================================================
# Alpha-Beta pruning method (Agent chooses move)
# =========================================================
def alphabeta(board, depth, alpha, beta, is_agent_turn):
    """
    Returns a numeric score using minimax with alpha-beta pruning.

    Depth = how many plies (half-moves) to look ahead.
    If depth reaches 0, we return utility(board).

    is_agent_turn=True  => AGENT (O) is moving => maximize score
    is_agent_turn=False => HUMAN (X) is moving => minimize score
    """
    # Return Utility Score For Terminal States (Win/Loss/Draw)
    if check_winner(board)!=None or is_draw(board)==True:
      return utility(board)

    # Return Utility Score For Non-Terminal States (Depth==0)
    if depth==0:
      return utility(board)

    # Maximizing player's turn: AGENT (is_agent_turn == True)
    if is_agent_turn == True:
      moves= get_available_moves(board)
      for idx in moves:
        board[idx]=AGENT
        max=utility(board)
        if max>alpha:
          alpha=max
          board[idx]=EMPTY
        depth-=1

    # Minimizing player's turn: HUMAN (is_agent_turn == False)
    if is_agent_turn == False:
      moves= get_available_moves(board)
      for idx in moves:
        board[idx]=HUMAN
        min=utility(board)
        if min<beta:
          beta=min
          board[idx]=EMPTY
        depth-=1

In [ ]:
def agent_choose_move(board):
    """
    Uses alpha-beta to select the best next move for the agent (O).

    NEW BEHAVIOR:
    - Searches up to LOOKAHEAD plies ahead (default 4).
    - If remaining moves < LOOKAHEAD, search only remaining moves.
    """
    moves = get_available_moves(board)
    if len(moves) == 0:
        return None

    # Limit depth to 4 plies (or fewer if game is near the end)
    depth = LOOKAHEAD
    if len(moves) < depth:
        depth = len(moves)

    best_move = moves[0]
    best_score = -999999

    alpha = -999999
    beta = 999999

    for mv in moves:
        board[mv] = AGENT

        # After agent move, next ply is human's turn
        score = alphabeta(board, depth - 1, alpha, beta, False)

        board[mv] = EMPTY

        if score > best_score:
            best_score = score
            best_move = mv

        if best_score > alpha:
            alpha = best_score

    return best_move


In [ ]:
# =========================================================
# 3) Method: alternate moves between human and agent
# =========================================================
def print_board(board):
    """
    Prints the board.
    Empty cells show their index to guide human input.
    """
    def show(i):
        if board[i] == EMPTY:
            return str(i)
        return board[i]

    print("\n")
    print(" " + show(0) + " | " + show(1) + " | " + show(2))
    print("---+---+---")
    print(" " + show(3) + " | " + show(4) + " | " + show(5))
    print("---+---+---")
    print(" " + show(6) + " | " + show(7) + " | " + show(8))
    print("\n")


def play_game():
    """
    Alternates turns until win/draw.
    Human plays X, Agent plays O.
    """
    board = [EMPTY] * 9
    human_turn = True  # X starts

    while True:
        print_board(board)

        w = check_winner(board)
        if w == HUMAN:
            print("Human (X) wins!")
            break
        if w == AGENT:
            print("Agent (O) wins!")
            break
        if is_draw(board):
            print("It's a draw!")
            break

        if human_turn:
            moves = get_available_moves(board)
            print("Available moves:", moves)

            while True:
                try:
                    mv = int(input("Enter your move (0..8): ").strip())
                except:
                    print("Invalid input. Please enter a number from 0 to 8.")
                    continue

                ok = place_mark(board, mv, HUMAN)
                if ok:
                    break
                else:
                    print("Illegal move. Try again.")
        else:
            mv = agent_choose_move(board)
            if mv is None:
                print("No moves left.")
                break
            place_mark(board, mv, AGENT)
            print("Agent plays:", mv)

        human_turn = not human_turn


if __name__ == "__main__":
    play_game()